# 04 - Bronze Validation

Cheap gate that runs after the three ingestion tasks: checks that every
expected Bronze table exists and has at least one row. If anything is
missing or empty, this task fails and the job stops here — Silver/Gold
never run against a broken or partial Bronze layer.

In [0]:
# Databricks notebook source
dbutils.widgets.text("catalog", "maven_test")
catalog = dbutils.widgets.get("catalog")

expected_tables = [
    f"{catalog}.bronze.calendar",
    f"{catalog}.bronze.promotions",
    f"{catalog}.bronze.regions",
    f"{catalog}.bronze.stores",
    f"{catalog}.bronze.returns",
    f"{catalog}.bronze.transactions",
    f"{catalog}.bronze.bronze_mongodb_customer_master",
    f"{catalog}.bronze.bronze_mongodb_product_master",
    f"{catalog}.bronze.kafka_online_orders_raw",
    f"{catalog}.bronze.kafka_inventory_raw",
]

In [0]:
results = []
for table in expected_tables:
    try:
        count = spark.table(table).count()
        status = "OK" if count > 0 else "EMPTY"
    except Exception as e:
        count = 0
        status = "MISSING"
    results.append({"table": table, "status": status, "row_count": count})
    print(f"{status:8s} {table} ({count} rows)")

failures = [r for r in results if r["status"] != "OK"]
if failures:
    raise RuntimeError(f"Bronze validation failed for: {[r['table'] for r in failures]}")

dbutils.notebook.exit("OK")